# ETL — Leitos Hospitalares

**Diferença em relação ao ETL:**
- No **ETL**, os dados são transformados em Python antes de entrar no banco.
- No **ELT**, os dados brutos entram no banco primeiro (schema `raw`), e todas as transformações acontecem dentro do PostgreSQL via **Views SQL** (schema `elt`).

**Fluxo:**
```
CSVs → Python (Pandas: Limpeza, Sets, Joins e Modelagem) → PostgreSQL (etl.dim_* + etl.fato_*)
```

## 1. Instalação e imports

In [ ]:
%pip install pandas sqlalchemy psycopg2-binary python-dotenv

In [80]:
import pandas as pd #principal biblioteca para manipulação dos CSVs e dados
import os
import csv 
from dotenv import load_dotenv #busca as variaveis do arquivo .env
from sqlalchemy import create_engine, text #conecta Python ao banco  

## 2 EXTRACT — extração e união dos dados

### 2.1 Funções auxiliares

Criando dicionario com os paths para os CSVs para ser usado futuramente

In [81]:
ARQUIVOS_LEITOS = {
    2023: "../database/data_raw/Leitos_2023.csv",
    2024: "../database/data_raw/Leitos_2024.csv",
    2025: "../database/data_raw/Leitos_2025.csv"
} 

Função para detectar automaticamente o separador dos arquivos de Anos diferentes, evitando erro na hora de concatenar

2023 e 2024 usam '**,**' | 2025 usa '**;**'

In [82]:
def detectar_separador(caminho_arquivo):
    with open(caminho_arquivo, "r", encoding="latin-1", newline="") as arquivo:
        amostra = arquivo.read(4096)

    separador = csv.Sniffer().sniff(amostra, delimiters=",;").delimiter

    return separador

Função que padroniza os nomes das colunas para facilitar consultas SQL

Exemplo:                    
    NOME_ESTABELECIMENTO → nome_estabelecimento                     
    LEITOS_EXISTENTES → leitos_existentes                   


In [83]:
def padronizar_nome_colunas(df):
    
    df = df.copy()

    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
    )

    return df

### 2.2 Função Principal

Função de leitura dos CSVs ultilizando as funções auxiliares

In [84]:
def ler_csv_leitos(caminho_arquivo):
    separador = detectar_separador(caminho_arquivo)

    df = pd.read_csv(
        caminho_arquivo,
        sep=separador,
        encoding="latin-1",
        low_memory=False
    )

    df = padronizar_nome_colunas(df)

    return df

### 2.3 Extraindo os arquivos CSVs de cada ano

In [85]:
bases = [] #lista para armazenar os DataFrames de cada ano

# Lendo os arquivos de leitos e armazenando na Lista
for caminho in ARQUIVOS_LEITOS.items():
    df_ano = ler_csv_leitos(caminho[1])
    bases.append(df_ano)

print("DataFrames lidos:")
for i, df in enumerate(bases):
    print(f"  - DataFrame {i+2023}: {df.shape[0]} linhas e {df.shape[1]} colunas")

DataFrames lidos:
  - DataFrame 2023: 84471 linhas e 34 colunas
  - DataFrame 2024: 85225 linhas e 34 colunas
  - DataFrame 2025: 86147 linhas e 35 colunas


### 2.4 Verificando diferencas entre os Datasets 


In [86]:
# Comparando as colunas entre os DataFrames de cada ano
dif_23_24 = set(bases[0].columns) - set(bases[1].columns)
print("Colunas que faltam em 2024 em relação a 2023:", dif_23_24)
new_23_24 = set(bases[1].columns) - set(bases[0].columns)
print("Colunas novas em 2024 em relação a 2023:", new_23_24)
print("-" * 50)

dif_23_25 = set(bases[0].columns) - set(bases[2].columns)
print("Colunas que faltam em 2025 em relação a 2023:", dif_23_25)
new_23_25 = set(bases[2].columns) - set(bases[0].columns)
print("Colunas novas em 2025 em relação a 2023:", new_23_25)
print("-" * 50)

dif_24_25 = set(bases[1].columns) - set(bases[2].columns)
print("Colunas que faltam em 2025 em relação a 2024:", dif_24_25)
new_24_25 = set(bases[2].columns) - set(bases[1].columns)
print("Colunas novas em 2025 em relação a 2024:", new_24_25)

Colunas que faltam em 2024 em relação a 2023: set()
Colunas novas em 2024 em relação a 2023: set()
--------------------------------------------------
Colunas que faltam em 2025 em relação a 2023: set()
Colunas novas em 2025 em relação a 2023: {'co_ibge'}
--------------------------------------------------
Colunas que faltam em 2025 em relação a 2024: set()
Colunas novas em 2025 em relação a 2024: {'co_ibge'}


### 2.5 Juntando os arquivos em um unico Database 

In [87]:
# Removendo colunas que não existem em todas as bases
bases[2].drop("co_ibge", axis=1, inplace=True)

# Juntando tudo em um único DataFrame
df_raw = pd.concat(bases, ignore_index=True, sort=False)
print(f"DataFrame unificado: {df_raw.shape[0]} linhas e {df_raw.shape[1]} colunas")

DataFrame unificado: 255843 linhas e 34 colunas


## 3. Analise Exploratoria dos Dados

### 3.1 Análises gerais:

In [88]:
df_raw.head()

,comp,regiao,uf,municipio,motivo_desabilitacao,cnes,nome_estabelecimento,razao_social,tp_gestao,co_tipo_unidade,...,uti_adulto_exist,uti_adulto_sus,uti_pediatrico_exist,uti_pediatrico_sus,uti_neonatal_exist,uti_neonatal_sus,uti_queimado_exist,uti_queimado_sus,uti_coronariana_exist,uti_coronariana_sus
0,202301,NORDESTE,PE,CABO DE SANTO AGOSTINHO,NaN,27,CASA DE SAUDE SANTA HELENA,CASA DE SAUDE E MATERNIDADE SANTA HELENA LTDA,M,5,...,0,0,0,0,0,0,0,0,0,0
1,202301,NORDESTE,PE,CABO DE SANTO AGOSTINHO,NaN,35,HOSPITAL MENDO SAMPAIO,PREFEITURA MUNICIPAL DO CABO DE SANTO AGOSTINHO,M,5,...,0,0,0,0,0,0,0,0,0,0
2,202301,NORDESTE,PE,CABO DE SANTO AGOSTINHO,NaN,94,MATERNIDADE PADRE GERALDO LEITE BASTOS,PREFEITURA MUNICIPAL DO CABO DE SANTO AGOSTINHO,M,7,...,0,0,0,0,0,0,0,0,0,0
3,202301,NORDESTE,PE,CABO DE SANTO AGOSTINHO,NaN,183,HOSPITAL SAMARITANO,SOCIEDADE HOSPITALAR SAMARITANO LTDA,M,5,...,5,0,0,0,0,0,0,0,0,0
4,202301,NORDESTE,PE,CABO DE SANTO AGOSTINHO,NaN,221,HOSPITAL SAO SEBASTIAO,CASA DE SAUDE E MATERNIDADE SAO SEBASTIAO LTDA,M,5,...,10,0,0,0,0,0,0,0,0,0


In [89]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 255843 entries, 0 to 255842
Data columns (total 34 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   comp                    255843 non-null  int64  
 1   regiao                  255843 non-null  str    
 2   uf                      255843 non-null  str    
 3   municipio               255843 non-null  str    
 4   motivo_desabilitacao    0 non-null       float64
 5   cnes                    255843 non-null  int64  
 6   nome_estabelecimento    255841 non-null  str    
 7   razao_social            255843 non-null  str    
 8   tp_gestao               255843 non-null  str    
 9   co_tipo_unidade         255843 non-null  int64  
 10  ds_tipo_unidade         255843 non-null  str    
 11  natureza_juridica       255843 non-null  int64  
 12  desc_natureza_juridica  255843 non-null  str    
 13  no_logradouro           255843 non-null  str    
 14  nu_endereco             255843 

### 3.2 Verificando os Tipos das colunas 

In [90]:
print(df_raw.dtypes)

comp                        int64
regiao                        str
uf                            str
municipio                     str
motivo_desabilitacao      float64
cnes                        int64
nome_estabelecimento          str
razao_social                  str
tp_gestao                     str
co_tipo_unidade             int64
ds_tipo_unidade               str
natureza_juridica           int64
desc_natureza_juridica        str
no_logradouro                 str
nu_endereco                   str
no_complemento                str
no_bairro                     str
co_cep                      int64
nu_telefone                   str
no_email                      str
leitos_existentes           int64
leitos_sus                  int64
uti_total_exist             int64
uti_total_sus               int64
uti_adulto_exist            int64
uti_adulto_sus              int64
uti_pediatrico_exist        int64
uti_pediatrico_sus          int64
uti_neonatal_exist          int64
uti_neonatal_s

Algumas Colunas que eram para ser **str** estão como **int64**

motivo_desabilitacao, cnes, co_tipo_unidade, natureza_juridica e co_cep

Trata-los posteriormente

### 3.3 Verificação de nulos

In [91]:
# Cria a tabela de diagnóstico de nulos
tabela_nulos = pd.DataFrame({
    'Total Nulos': df_raw.isna().sum(),
    'Porcentagem %': ((df_raw.isna().sum() / len(df_raw)) * 100).round(2)
}).sort_values(by='Total Nulos', ascending=False)

print(tabela_nulos)

                        Total Nulos  Porcentagem %
motivo_desabilitacao         255843         100.00
no_complemento               207694          81.18
no_email                      84786          33.14
nu_telefone                   29334          11.47
nome_estabelecimento              2           0.00
comp                              0           0.00
regiao                            0           0.00
razao_social                      0           0.00
cnes                              0           0.00
municipio                         0           0.00
co_tipo_unidade                   0           0.00
ds_tipo_unidade                   0           0.00
desc_natureza_juridica            0           0.00
natureza_juridica                 0           0.00
no_logradouro                     0           0.00
nu_endereco                       0           0.00
tp_gestao                         0           0.00
uf                                0           0.00
co_cep                         

Algumas Colunas estão com muitos nulos, verificar se necessário a permanencia delas

### 3.4 Verificação de duplicadas

In [92]:
# Valores Duplicados
df_duplicados = df_raw[df_raw.duplicated(keep=False)]
print(f"Total de linhas duplicadas: {len(df_duplicados)}")

Total de linhas duplicadas: 0


### 3.5 Verificação de números únicos

In [93]:
print("Tabela de valores únicos por coluna:")
print(df_raw.nunique().sort_values(ascending=True))

Tabela de valores únicos por coluna:
motivo_desabilitacao         0
tp_gestao                    3
desc_natureza_juridica       3
regiao                       5
co_tipo_unidade              5
ds_tipo_unidade              5
uti_queimado_sus             8
uti_queimado_exist          15
uti_coronariana_sus         19
uf                          27
uti_coronariana_exist       30
natureza_juridica           32
comp                        36
uti_neonatal_sus            39
uti_pediatrico_sus          40
uti_pediatrico_exist        51
uti_neonatal_exist          56
uti_adulto_sus              84
uti_total_sus              116
uti_adulto_exist           122
uti_total_exist            155
leitos_sus                 557
leitos_existentes          625
no_complemento             994
nu_endereco               1938
no_bairro                 2476
municipio                 3473
no_email                  6063
razao_social              6731
no_logradouro             7047
co_cep                    7198
cn

In [94]:
#Função para mostrar os valores únicos de uma coluna específica
def valores_unicos(df, coluna):
    valores_unicos = df[coluna].unique()
    print(f"Valores únicos na coluna '{coluna}':")
    print(valores_unicos)
    print("-" * 50)

In [95]:
# Verificando os valores únicos de algumas colunas específicas
valores_unicos(df_raw, "tp_gestao")
valores_unicos(df_raw, "desc_natureza_juridica")
valores_unicos(df_raw, "regiao")
valores_unicos(df_raw, "ds_tipo_unidade")

Valores únicos na coluna 'tp_gestao':
<StringArray>
['M', 'E', 'D']
Length: 3, dtype: str
--------------------------------------------------
Valores únicos na coluna 'desc_natureza_juridica':
<StringArray>
['HOSPITAL_PRIVADO', 'HOSPITAL_Pï¿½BLICO', 'HOSPITAL_FILANTRï¿½PICO']
Length: 3, dtype: str
--------------------------------------------------
Valores únicos na coluna 'regiao':
<StringArray>
['NORDESTE', 'NORTE', 'SUDESTE', 'CENTRO-OESTE', 'SUL']
Length: 5, dtype: str
--------------------------------------------------
Valores únicos na coluna 'ds_tipo_unidade':
<StringArray>
[              'HOSPITAL GERAL',       'HOSPITAL ESPECIALIZADO',
         'PRONTO SOCORRO GERAL',                'UNIDADE MISTA',
 'PRONTO SOCORRO ESPECIALIZADO']
Length: 5, dtype: str
--------------------------------------------------


In [96]:
valores_unicos(df_raw, "no_complemento")

Valores únicos na coluna 'no_complemento':
<StringArray>
[                   nan,             'BLOCO 01',            '1 SUBSOLO',
    'FAZENDA GRANDE II',        'CAJAZEIRAS II',            'SALA ABSJ',
          'VILA RENATA',           'SAO JULIAO',        'SETOR CENTRAL',
               'QNM 27',
 ...
                '1 037', 'ANEXO HOSPITAL DE CA', 'A POLICLIN IBIRAPUER',
      'QD G 2 LT 40 42',            'QD112 L16',         'TERREO BLOCO',
  'CONJ B BLOCO I E II',                  '209',              'LOTE 7A',
 '886 88 90 E 92 SUP R']
Length: 995, dtype: str
--------------------------------------------------


Na coluna tp_gestão não fica muito claro o que cada sigla significa, adicionar uma descrição posteriormente

Percebe-se que a coluna no_complemento não tem um padrão e não agraga muito valor para nosso dataset, retiraremos no futuro

## 4. Tranformação dos dados

### 4.1 Definir as colunas em formas que serão tratadas

In [97]:
COLUNAS_TEXTO = [ #Colunas descritivas de texto e que serão mantidas como tal
    "regiao",
    "uf",
    "municipio",
    "nome_estabelecimento",
    "razao_social",
    "tp_gestao",
    "ds_tipo_unidade",
    "desc_natureza_juridica",
    "no_logradouro",
    "nu_endereco",
    "no_complemento",
    "no_bairro",
    "nu_telefone",
    "no_email"
]

COLUNAS_CODIGO = { #Colunas que são do tipo Int e que serão convertidas para string
    "comp",
    "cnes",
    "co_tipo_unidade",
    "natureza_juridica",
    "co_cep"
}

METRICAS_LEITOS = [ #Colunas que são do tipo Int e que serão mantidas como tal
    "leitos_existentes",
    "leitos_sus",
    "uti_total_exist",
    "uti_total_sus",
    "uti_adulto_exist",
    "uti_adulto_sus",
    "uti_pediatrico_exist",
    "uti_pediatrico_sus",
    "uti_neonatal_exist",
    "uti_neonatal_sus",
    "uti_queimado_exist",
    "uti_queimado_sus",
    "uti_coronariana_exist",
    "uti_coronariana_sus"
]

### 4.2 Função principal do tratamento

In [98]:
def preparar_leitos(df_raw):
    
    # Criamos uma cópia para não alterar diretamente o DataFrame original.
    df = df_raw.copy()

    #Tirando colunas com muitos nulos que não agragam valor
    df.drop("motivo_desabilitacao", axis=1, inplace=True)
    df.drop("no_complemento", axis=1, inplace=True)

    # Padronização das colunas de texto.
    # Aqui removemos espaços extras, transformamos tudo em minusculo
    # e substituímos valores nulos por "NAO_INFORMADO".
    for col in COLUNAS_TEXTO:
        if col in df.columns:
            df[col] = df[col].fillna("nao_informado")
            df[col] = (
                df[col]
                .astype("str")
                .str.strip()
                .str.lower()
            )

    # Padronização das colunas de códigos.
    # Esses campos são tratados como texto, não como número.
    # Isso preserva zeros à esquerda e evita interpretações erradas.
    for col in COLUNAS_CODIGO:
        if col in df.columns:
            df[col] = (
                df[col]
                .astype("str")
                .str.strip()
                .str.replace(r"\.0$", "", regex=True)
            )
    # Padronização dos códigos especificos.
    # zfill completa com zeros à esquerda.
    # Exemplo: "123" vira "0000123" se o tamanho esperado for 7.
    df["comp"] = df["comp"].str.zfill(6)
    df["cnes"] = df["cnes"].str.zfill(7)
    df["co_tipo_unidade"] = df["co_tipo_unidade"].str.zfill(2)
    df["natureza_juridica"] = df["natureza_juridica"].str.zfill(4)
    df["co_cep"] = df["co_cep"].str.zfill(8)

    # Padronização das colunas numéricas.
    # Converte as métricas de leitos e UTIs para número inteiro.
    # Se algum valor vier inválido, ele vira nulo com errors="coerce".
    for col in METRICAS_LEITOS:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype(int)


    #Renomeando as colunas para melhor entendimento e padronização
    novos_nomes = {
    "comp": "competencia",
    "ds_tipo_unidade": "desc_tipo_unidade",
    "no_logradouro": "logradouro",
    "tp_gestao": "tipo_gestao",
    "nu_endereco":"num_endereco",
    "no_bairro":"bairro",
    "nu_telefone":"telefone",
    "no_email":"email",
    "co_tipo_unidade":"cod_tipo_unidade",
    "co_cep":"cep",
    }
    df = df.rename(columns=novos_nomes)

    return df

### 4.3 Aplicando as Transformações e criando um novo Dataframe

In [99]:
# Chamando a função de preparação dos dados
df_preparado = preparar_leitos(df_raw)

print(f"DataFrame preparado: {df_preparado.shape[0]} linhas e {df_preparado.shape[1]} colunas")
df_preparado.head()

DataFrame preparado: 255843 linhas e 32 colunas


,competencia,regiao,uf,municipio,cnes,nome_estabelecimento,razao_social,tipo_gestao,cod_tipo_unidade,desc_tipo_unidade,...,uti_adulto_exist,uti_adulto_sus,uti_pediatrico_exist,uti_pediatrico_sus,uti_neonatal_exist,uti_neonatal_sus,uti_queimado_exist,uti_queimado_sus,uti_coronariana_exist,uti_coronariana_sus
0,202301,nordeste,pe,cabo de santo agostinho,0000027,casa de saude santa helena,casa de saude e maternidade santa helena ltda,m,05,hospital geral,...,0,0,0,0,0,0,0,0,0,0
1,202301,nordeste,pe,cabo de santo agostinho,0000035,hospital mendo sampaio,prefeitura municipal do cabo de santo agostinho,m,05,hospital geral,...,0,0,0,0,0,0,0,0,0,0
2,202301,nordeste,pe,cabo de santo agostinho,0000094,maternidade padre geraldo leite bastos,prefeitura municipal do cabo de santo agostinho,m,07,hospital especializado,...,0,0,0,0,0,0,0,0,0,0
3,202301,nordeste,pe,cabo de santo agostinho,0000183,hospital samaritano,sociedade hospitalar samaritano ltda,m,05,hospital geral,...,5,0,0,0,0,0,0,0,0,0
4,202301,nordeste,pe,cabo de santo agostinho,0000221,hospital sao sebastiao,casa de saude e maternidade sao sebastiao ltda,m,05,hospital geral,...,10,0,0,0,0,0,0,0,0,0


## 5. Verificando regras de negocios

### 5.1 Verificar duplicidade do grão

In [100]:
duplicadas = df_preparado.duplicated(["competencia", "cnes"]).sum()

print("Duplicidades por competencia + cnes:", duplicadas)

Duplicidades por competencia + cnes: 0




O grão esperado da tabela fato é uma linha por estabelecimento de saúde, identificado pelo CNES, em uma competência mensal, identificada pela coluna COMP.                                                          

Se der 0, significa que cada estabelecimento aparece uma única vez por mês.
Isso confirma o grão da fato.


### 5.2 Verificar se leitos SUS não ultrapassam leitos existentes


In [101]:
erro_leitos_sus = (
    df_preparado["leitos_sus"] > df_preparado["leitos_existentes"]
).sum()

print("Linhas com leitos_sus > leitos_existentes:", erro_leitos_sus)

Linhas com leitos_sus > leitos_existentes: 0


LEITOS_SUS não deveria ser maior que LEITOS_EXISTENTES.
Essa regra valida a consistência da base.

### 5.3 Verificar se UTI SUS não ultrapassa UTI existente

In [102]:
erro_uti_sus = (
    df_preparado["uti_total_sus"] > df_preparado["uti_total_exist"]
).sum()

print("Linhas com uti_total_sus > uti_total_exist:", erro_uti_sus)

Linhas com uti_total_sus > uti_total_exist: 0


### 5.4 Verificar se UTI total bate com a soma dos tipos de UTI

In [103]:
soma_uti_existente = (
    df_preparado["uti_adulto_exist"]
    + df_preparado["uti_pediatrico_exist"]
    + df_preparado["uti_neonatal_exist"]
    + df_preparado["uti_queimado_exist"]
    + df_preparado["uti_coronariana_exist"]
)

erro_soma_uti_existente = (
    df_preparado["uti_total_exist"] != soma_uti_existente
).sum()

print("Linhas com divergência na soma das UTIs existentes:", erro_soma_uti_existente)

Linhas com divergência na soma das UTIs existentes: 0


### 5.5 Verificar se UTI sus total bate com a soma dos tipos de UTI sus

In [104]:
soma_uti_sus = (
    df_preparado["uti_adulto_sus"]
    + df_preparado["uti_pediatrico_sus"]
    + df_preparado["uti_neonatal_sus"]
    + df_preparado["uti_queimado_sus"]
    + df_preparado["uti_coronariana_sus"]
)

erro_soma_uti_sus = (
    df_preparado["uti_total_sus"] != soma_uti_sus
).sum()

print("Linhas com divergência na soma das UTIs SUS:", erro_soma_uti_sus)

Linhas com divergência na soma das UTIs SUS: 0


## 6. Criação das dimensões

### 6.1 Função auxiliar:

In [105]:
# Função para criar dimensões a partir de uma lista com as colunas específicas
def criar_dimensao(df, colunas, nome_id):
    dim = ( # Adiciona as colunas selecionadas, remove duplicatas, ordena e reseta o índice
        df[colunas]
        .drop_duplicates()
        .sort_values(colunas, na_position="last")
        .reset_index(drop=True)
    )

    dim.insert(0, nome_id, range(1, len(dim) + 1)) #Cria uma coluna de ID sequencial começando em 1

    return dim

### 6.2 Dimensão tempo

In [106]:
# Dicionário para transformar número do mês em nome do mês.
mapa_meses = {
    1: "janeiro",
    2: "fevereiro",
    3: "marco",
    4: "abril",
    5: "maio",
    6: "junho",
    7: "julho",
    8: "agosto",
    9: "setembro",
    10: "outubro",
    11: "novembro",
    12: "dezembro"
}

# Criando as colunas da dimensao tempo a partir da coluna COMP
# Cria coluna de data usando o ano e mês da competência, ultilizando o dia 01 como padrão
df_preparado["data_competencia"] = pd.to_datetime( 
    df_preparado["competencia"] .astype(str) + "01", format="%Y%m%d", errors="coerce")
# Cria as colunas de ano, mês e trimestre a partir da data de competência
df_preparado["ano"] = df_preparado["data_competencia"].dt.year.astype("Int64")
df_preparado["mes"] = df_preparado["data_competencia"].dt.month.astype("Int64")
df_preparado["trimestre"] = df_preparado["data_competencia"].dt.quarter.astype("Int64")
# Cria a coluna de nome do mês usando o mapa_meses para converter o número do mês em texto
df_preparado["nome_mes"] = df_preparado["mes"].map(mapa_meses)


# Criando a dimensão tempo usando as colunas criadas
dim_tempo = criar_dimensao(
    df_preparado, 
    [
        "competencia", 
        "data_competencia", 
        "ano", 
        "mes", 
        "nome_mes", 
        "trimestre"
    ], 
    "id_tempo"
    )

dim_tempo.head()


,id_tempo,competencia,data_competencia,ano,mes,nome_mes,trimestre
0,1,202301,2023-01-01,2023,1,janeiro,1
1,2,202302,2023-02-01,2023,2,fevereiro,1
2,3,202303,2023-03-01,2023,3,marco,1
3,4,202304,2023-04-01,2023,4,abril,2
4,5,202305,2023-05-01,2023,5,maio,2


Essa dimensão responde perguntas como:          
Quantos leitos havia em janeiro de 2025?            
Como os leitos evoluíram mês a mês?         
Qual foi o total por ano?       

### 6.3 Dimensão estabelecimento de saúde

In [108]:
print(df_preparado.columns.tolist())

['competencia', 'regiao', 'uf', 'municipio', 'cnes', 'nome_estabelecimento', 'razao_social', 'tipo_gestao', 'cod_tipo_unidade', 'desc_tipo_unidade', 'natureza_juridica', 'desc_natureza_juridica', 'logradouro', 'num_endereco', 'bairro', 'cep', 'telefone', 'email', 'leitos_existentes', 'leitos_sus', 'uti_total_exist', 'uti_total_sus', 'uti_adulto_exist', 'uti_adulto_sus', 'uti_pediatrico_exist', 'uti_pediatrico_sus', 'uti_neonatal_exist', 'uti_neonatal_sus', 'uti_queimado_exist', 'uti_queimado_sus', 'uti_coronariana_exist', 'uti_coronariana_sus', 'data_competencia', 'ano', 'mes', 'trimestre', 'nome_mes']


In [109]:
dim_estabelecimento_saude = criar_dimensao(
    df_preparado,
    [
        "cnes",
        "nome_estabelecimento",
        "razao_social",
        "logradouro",
        "num_endereco",
        "bairro",
        "cep",
        "regiao",
        "uf",
        "municipio",
        "telefone",
        "email"
    ],
    "id_estabelecimento"
)

dim_estabelecimento_saude.head()

,id_estabelecimento,cnes,nome_estabelecimento,razao_social,logradouro,num_endereco,bairro,cep,regiao,uf,municipio,telefone,email
0,1,0000027,casa de saude santa helena,casa de saude e maternidade santa helena ltda,avn presidente getulio vargas,428,centro,54505560,nordeste,pe,cabo de santo agostinho,(81)35210355,nao_informado
1,2,0000035,hospital mendo sampaio,prefeitura municipal do cabo de santo agostinho,br 101 sul km 33,s/n,charneca,54535430,nordeste,pe,cabo de santo agostinho,(81)35210857,nao_informado
2,3,0000094,maternidade padre geraldo leite bastos,prefeitura municipal do cabo de santo agostinho,br 101 sul km 23,s/n,ponte dos carvalhos,54510000,nordeste,pe,cabo de santo agostinho,(81)35221626,nao_informado
3,4,0000183,hospital samaritano,sociedade hospitalar samaritano ltda,rua severino bezerra marques,40,centro,54510460,nordeste,pe,cabo de santo agostinho,(81)35210109,nao_informado
4,5,0000221,hospital sao sebastiao,casa de saude e maternidade sao sebastiao ltda,avn presidente getulio vargas,864,centro,54505560,nordeste,pe,cabo de santo agostinho,(81)35214150,adm@hospitalsaosebastiao.com.br


CNES identifica o estabelecimento.                                                  
Nome, razão social e endereço descrevem esse estabelecimento.

Essa dimensão tambem permite consultas por:                
região          
estado          
município                                                             

### 6.4. Dimensão tipo de unidade

In [110]:
dim_tipo_unidade = criar_dimensao(
    df_preparado,
    [
        "cod_tipo_unidade",
        "desc_tipo_unidade"
    ],
    "id_tipo_unidade"
)

dim_tipo_unidade.head()

,id_tipo_unidade,cod_tipo_unidade,desc_tipo_unidade
0,1,05,hospital geral
1,2,07,hospital especializado
2,3,15,unidade mista
3,4,20,pronto socorro geral
4,5,21,pronto socorro especializado


Exemplos de tipo de unidade:

HOSPITAL GERAL,                 
HOSPITAL ESPECIALIZADO,                 
PRONTO SOCORRO GERAL,               
UNIDADE MISTA               

### 6.5. Dimensão jurídica

In [111]:
dim_natureza_juridica = criar_dimensao(
    df_preparado,
    [
        "natureza_juridica",
        "desc_natureza_juridica"
    ],
    "id_natureza_juridica"
)

dim_natureza_juridica.head()

,id_natureza_juridica,natureza_juridica,desc_natureza_juridica
0,1,1015,hospital_pï¿½blico
1,2,1023,hospital_pï¿½blico
2,3,1031,hospital_pï¿½blico
3,4,1104,hospital_pï¿½blico
4,5,1112,hospital_pï¿½blico


Essa dimensão ajuda a comparar:

hospital público    
hospital privado    
hospital filantrópico   
outras categorias       

### 6.6. Dimensão gestão

In [112]:
   
# Dicionário de domínio para o tipo de gestão.
mapa_gestao = {
    "m": "municipal",
    "e": "estadual",
    "d": "dupla"
}

# Cria a coluna descrição da gestão.
# Valores que não estiverem no dicionário viram NAO_INFORMADO.
df_preparado["desc_gestao"] = (
    df_preparado["tipo_gestao"]
    .fillna("nao informado")  
    .map(mapa_gestao)     
)

# Criando a dimensão gestão usando as colunas de tipo e descrição da gestão
dim_gestao = criar_dimensao(
    df_preparado,
    [
        "tipo_gestao", 
        "desc_gestao"
    ],
    "id_gestao"
)

dim_gestao.head()

,id_gestao,tipo_gestao,desc_gestao
0,1,d,dupla
1,2,e,estadual
2,3,m,municipal


Essa dimensão ajuda a identificar a esfera de governo responsável pela regulação e comando operacional dos leitos hospitalares

## 7 Criação da tabela fato

A fato será:                

fato_leitos_mensais         

O grão dela é:          

uma linha por estabelecimento de saúde em uma competência mensal                

In [113]:
fato = df_preparado.copy()

fato = fato.merge(
    dim_tempo[["id_tempo", "competencia"]],
    on="competencia",
    how="left"
)

fato = fato.merge( 
    dim_estabelecimento_saude[
        [
            "id_estabelecimento",
            "cnes",
            "nome_estabelecimento",
            "razao_social",
            "logradouro",
            "num_endereco",
            "bairro",
            "cep",
            "regiao",
            "uf",
            "municipio",
            "telefone",
            "email"
        ]
    ],
    on=[
        "cnes",
        "nome_estabelecimento",
        "razao_social",
        "logradouro",
        "num_endereco",
        "bairro",
        "cep",
        "regiao",
        "uf",
        "municipio",
        "telefone",
        "email"
    ],
    how="left"
)

fato = fato.merge(
    dim_tipo_unidade[
        [
            "id_tipo_unidade",
            "cod_tipo_unidade",
            "desc_tipo_unidade"
        ]
    ],
    on=["cod_tipo_unidade", "desc_tipo_unidade"],
    how="left"
)

fato = fato.merge(
    dim_natureza_juridica[
        [
            "id_natureza_juridica",
            "natureza_juridica",
            "desc_natureza_juridica"
        ]
    ],
    on=["natureza_juridica", "desc_natureza_juridica"],
    how="left"
)

fato = fato.merge(
    dim_gestao[["id_gestao", "tipo_gestao", "desc_gestao"]],
    on=["tipo_gestao", "desc_gestao"],
    how="left"
)

Explicação:             
                        
Cada merge procura o registro correspondente na dimensão.               
Depois disso, a fato recebe os IDs:             
id_tempo                        
id_estabelecimento                      
id_localidade                               
id_tipo_unidade             
id_natureza_juridica                    
id_gestao                               

### 7.2  Selecionar as colunas finais da fato

In [114]:
# Selecionando apenas as colunas de chaves das dimensões e as métricas de leitos para a tabela fato
colunas_fato = [
    "id_tempo",
    "id_estabelecimento",
    "id_tipo_unidade",
    "id_natureza_juridica",
    "id_gestao"
] + METRICAS_LEITOS

# Criando o DataFrame final da tabela fato com as colunas selecionadas
fato_leitos_mensais = fato[colunas_fato]
fato_leitos_mensais.head()

,id_tempo,id_estabelecimento,id_tipo_unidade,id_natureza_juridica,id_gestao,leitos_existentes,leitos_sus,uti_total_exist,uti_total_sus,uti_adulto_exist,uti_adulto_sus,uti_pediatrico_exist,uti_pediatrico_sus,uti_neonatal_exist,uti_neonatal_sus,uti_queimado_exist,uti_queimado_sus,uti_coronariana_exist,uti_coronariana_sus
0,1,1,1,20,3,64,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1,2,1,14,3,19,19,0,0,0,0,0,0,0,0,0,0,0,0
2,1,3,2,14,3,18,18,0,0,0,0,0,0,0,0,0,0,0,0
3,1,4,1,20,3,59,0,5,0,5,0,0,0,0,0,0,0,0,0
4,1,5,1,20,3,47,0,10,0,10,0,0,0,0,0,0,0,0,0


### 7.3 Verificar se algum id ficou nulo

In [115]:
fato_leitos_mensais[
    [
        "id_tempo",
        "id_estabelecimento",
        "id_tipo_unidade",
        "id_natureza_juridica",
        "id_gestao"
    ]
].isna().sum()

id_tempo                0
id_estabelecimento      0
id_tipo_unidade         0
id_natureza_juridica    0
id_gestao               0
dtype: int64

## 8 Carregar no PostgreSQL

### 8.1 Configurando a porta de comunicação

In [ ]:
load_dotenv()

user = os.getenv('DB_USER')
password = os.getenv('DB_PASSWORD')
db = os.getenv('DB_NAME')
port = os.getenv('DB_PORT')
host = os.getenv('DB_HOST')

DATABASE_URL = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{db}"

engine = create_engine(DATABASE_URL)

### 8.2 Criando schemas

In [ ]:
with engine.begin() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS etl;"))

### 8.3 Enviar tabelas:

In [ ]:
dim_tempo.to_sql(
    "dim_tempo",
    con=engine,
    schema="etl",
    if_exists="replace",
    index=False
)

dim_estabelecimento_saude.to_sql(
    "dim_estabelecimento_saude",
    con=engine,
    schema="etl",
    if_exists="replace",
    index=False
)

dim_tipo_unidade.to_sql(
    "dim_tipo_unidade",
    con=engine,
    schema="etl",
    if_exists="replace",
    index=False
)

dim_natureza_juridica.to_sql(
    "dim_natureza_juridica",
    con=engine,
    schema="etl",
    if_exists="replace",
    index=False
)

dim_gestao.to_sql(
    "dim_gestao",
    con=engine,
    schema="etl",
    if_exists="replace",
    index=False
)

fato_leitos_mensais.to_sql(
    "fato_leitos_mensais",
    con=engine,
    schema="etl",
    if_exists="replace",
    index=False
)